# followup posterior evaluation

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Follow-up posterior evaluation and paired bootstrap

This notebook evaluates the saved follow-up NPE models without fitting them again. The
purpose is to export observation-level posterior summaries and determine whether the
differences between feature representations remain after resampling connected host
components. The same data definitions, scalers, feature order, and held-out indices used by
the saved runs are included.

The notebook produces three outputs. First, it saves posterior draws and one row of posterior
summary statistics for every evaluated pair. Second, it recomputes the pooled, same-host, and
different-host metrics using a common number of posterior draws. Finally, it performs paired
connected-host bootstrap comparisons between models with the same target. A paired bootstrap
is used because every model is evaluated on the same systems. This reduces variation caused
by changing the evaluation population.

No NPE training is performed in this notebook. The domain-classifier and common-support
experiment is also kept separate because it answers a different question.


## 1. Roihu environment and configuration

The saved models were created with the Roihu `python-pytorch/2.13` module and the `lgenv`
environment. Model objects saved with pickle are sensitive to package versions. Therefore,
the notebook stops if the important versions differ from the versions recorded by the runs.

The default uses 500 posterior draws. This is larger than the 200 draws used by the original
run manifests and reduces finite-draw variation in coverage and rank statistics. The new
metrics should therefore be compared across models within this notebook rather than expected
to match every manifest value exactly.


In [ ]:
from __future__ import annotations

import gc
import importlib.metadata as metadata
import json
import os
import pickle
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch


def find_project_root() -> Path:
    candidates = []
    if os.environ.get("LG_PROJECT"):
        candidates.append(Path(os.environ["LG_PROJECT"]))
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "src" / "lg_npe").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find the FLAMINGO project. Set LG_PROJECT to the repository path."
    )


PROJECT = find_project_root()
sys.path.insert(0, str(PROJECT / "src"))

DRAW_COUNT = 500
EVALUATION_BATCH_SIZE = 256
BOOTSTRAP_REPEATS = 2000
BOOTSTRAP_CHUNK_SIZE = 50
RANDOM_SEED = 260917
SAVE_POSTERIOR_DRAWS = True

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT = PROJECT / "out" / "mass_comparison" / "posterior_evaluation" / timestamp
OUTPUT.mkdir(parents=True, exist_ok=False)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("This notebook must be executed on a Roihu GPU node.")

print(f"Project: {PROJECT}")
print(f"Output:  {OUTPUT}")
print(f"Device:  {torch.cuda.get_device_name(0)}")

In [ ]:
expected_versions = {
    "python_major_minor": "3.12",
    "sbi": "0.27.0",
    "scikit-learn": "1.9.0",
    "torch_prefix": "2.13.0",
}

actual_versions = {
    "python": platform.python_version(),
    "sbi": metadata.version("sbi"),
    "scikit-learn": sklearn.__version__,
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}

problems = []
if not actual_versions["python"].startswith(expected_versions["python_major_minor"] + "."):
    problems.append(f"Python {actual_versions['python']}")
if actual_versions["sbi"] != expected_versions["sbi"]:
    problems.append(f"sbi {actual_versions['sbi']}")
if actual_versions["scikit-learn"] != expected_versions["scikit-learn"]:
    problems.append(f"scikit-learn {actual_versions['scikit-learn']}")
if not actual_versions["torch"].startswith(expected_versions["torch_prefix"]):
    problems.append(f"torch {actual_versions['torch']}")

print(json.dumps(actual_versions, indent=2))
if problems:
    raise RuntimeError(
        "The active kernel does not reproduce the saved-model environment: "
        + ", ".join(problems)
    )

## 2. Saved-run registry

The registry below contains the baseline and the eight new scientific runs. The run paths are
fixed because these directories contain the models whose results were already reported. The
notebook checks the experiment identifier, feature names, target, and saved split before any
posterior samples are drawn.


In [ ]:
RUNS = {
    "m200c_bound_ratio": "out/mass_comparison/stellar_features/m200c_dynamics_bound_subhalo_ratio_baseline_replication/20260917T092350Z",
    "m200c_dynamics": "out/mass_comparison/stellar_features/m200c_dynamics_no_mass/20260917T094727Z",
    "m200c_stellar_individual": "out/mass_comparison/stellar_features/m200c_dynamics_stellar_individual/20260917T094931Z",
    "m200c_stellar_ratio": "out/mass_comparison/stellar_features/m200c_dynamics_stellar_ratio/20260917T094527Z",
    "m200c_stellar_sum": "out/mass_comparison/stellar_features/m200c_dynamics_stellar_sum/20260917T095930Z",
    "bound_sum_dynamics": "out/mass_comparison/bound_mass_target/bound_subhalo_sum_dynamics_no_mass/20260917T102331Z",
    "bound_sum_stellar_individual": "out/mass_comparison/bound_mass_target/bound_subhalo_sum_dynamics_stellar_individual/20260917T103342Z",
    "bound_sum_stellar_ratio": "out/mass_comparison/bound_mass_target/bound_subhalo_sum_dynamics_stellar_ratio/20260917T102118Z",
    "bound_sum_stellar_sum": "out/mass_comparison/bound_mass_target/bound_subhalo_sum_dynamics_stellar_sum/20260917T103841Z",
}
RUNS = {label: PROJECT / relative for label, relative in RUNS.items()}

required_files = {"model.pkl", "run_manifest.json", "split_indices.npz"}
for label, run_dir in RUNS.items():
    missing = sorted(name for name in required_files if not (run_dir / name).is_file())
    if missing:
        raise FileNotFoundError(f"{label} is missing: {missing}")

print(f"Registered {len(RUNS)} scientific runs.")

## 3. Data reconstruction and validation

Each design matrix is reconstructed from the saved configuration. The source scaler stored in
the model is applied to both FLAMINGO and TNG, as in the training pipeline. The saved indices
are used directly rather than making a new split. This is important because changing the split
would make the paired model comparison invalid.


In [ ]:
from lg_npe.data import load_flamingo_pair_table, load_tng_pair_table
from lg_npe.evaluation import posterior_metrics
from lg_npe.features import build_design_matrix
from lg_npe.inference import posterior_draws
from lg_npe.splitting import pair_groups

source_table = load_flamingo_pair_table(PROJECT)
transfer_table = load_tng_pair_table(PROJECT)


def reconstruct_run(label: str, run_dir: Path) -> dict:
    manifest = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
    with (run_dir / "model.pkl").open("rb") as stream:
        model = pickle.load(stream)
    with np.load(run_dir / "split_indices.npz") as saved:
        split = {name: saved[name] for name in saved.files}

    config = model["config"]
    source, names, target, source_counts = build_design_matrix(
        source_table.frame,
        target=config["target"],
        feature_block=config["feature_block"],
        mass_feature=config["mass_feature"],
        validity_policy=config["validity_policy"],
    )
    transfer, transfer_names, transfer_target, transfer_counts = build_design_matrix(
        transfer_table.frame,
        target=config["target"],
        feature_block=config["feature_block"],
        mass_feature=config["mass_feature"],
        validity_policy=config["validity_policy"],
    )

    if names != transfer_names or target != transfer_target:
        raise RuntimeError(f"Source and transfer definitions disagree for {label}.")
    if names != list(model["feature_names"]):
        raise RuntimeError(f"Feature order disagrees with the saved model for {label}.")
    if target != model["target"]:
        raise RuntimeError(f"Target disagrees with the saved model for {label}.")
    if len(split["source_groups"]) != len(source):
        raise RuntimeError(f"Saved source groups have the wrong length for {label}.")
    if split["test"].max() >= len(source) or split["transfer"].max() >= len(transfer):
        raise RuntimeError(f"Saved indices are outside the reconstructed data for {label}.")

    return {
        "label": label,
        "run_dir": run_dir,
        "manifest": manifest,
        "model": model,
        "config": config,
        "feature_names": names,
        "target": target,
        "source": source,
        "transfer": transfer,
        "split": split,
        "source_counts": source_counts,
        "transfer_counts": transfer_counts,
    }


validated = {}
for label, run_dir in RUNS.items():
    run = reconstruct_run(label, run_dir)
    validated[label] = run
    print(
        f"{label:30s} target={run['config']['target']:18s} "
        f"features={len(run['feature_names'])} "
        f"source_test={len(run['split']['test']):,} "
        f"transfer={len(run['split']['transfer']):,}"
    )

## 4. Posterior export

Posterior samples are generated for the held-out FLAMINGO population and the complete TNG
evaluation population. For every pair, the exported table contains the posterior median,
mean, standard deviation, credible intervals, rank, standardized error, and host membership.
The raw posterior draws are included in a compressed archive. This allows calibration to be
recomputed later without loading the model again.

The random seed is fixed separately for each run and population. This makes the export
reproducible, although posterior sampling still introduces finite-draw variation.


In [ ]:
def set_sampling_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def metrics_by_host(samples: np.ndarray, truth: np.ndarray, same_host: np.ndarray) -> dict:
    result = {"pooled": posterior_metrics(samples, truth)}
    if same_host.any():
        result["same_host"] = posterior_metrics(samples[:, same_host], truth[same_host])
    if (~same_host).any():
        result["different_host"] = posterior_metrics(samples[:, ~same_host], truth[~same_host])
    return result


def per_pair_summary(
    samples: np.ndarray,
    frame: pd.DataFrame,
    indices: np.ndarray,
    target: str,
    groups: np.ndarray,
) -> pd.DataFrame:
    selected = frame.iloc[indices]
    truth = selected[target].to_numpy(np.float32)
    q025, q16, q50, q84, q975 = np.percentile(samples, [2.5, 16, 50, 84, 97.5], axis=0)
    mean = samples.mean(axis=0)
    std = samples.std(axis=0)
    residual = q50 - truth
    beta = np.divide(
        np.abs(residual), std, out=np.full_like(std, np.inf), where=std > 0
    )
    ranks = (samples < truth[None, :]).sum(axis=0)

    return pd.DataFrame({
        "pair_index": selected["pair_index"].to_numpy(),
        "host_i": selected["host_i"].to_numpy(),
        "host_j": selected["host_j"].to_numpy(),
        "connected_component": groups[indices],
        "same_host": selected["same_host"].to_numpy(bool),
        "truth": truth,
        "posterior_mean": mean,
        "posterior_median": q50,
        "posterior_std": std,
        "q025": q025,
        "q16": q16,
        "q84": q84,
        "q975": q975,
        "residual": residual,
        "beta": beta,
        "rank": ranks,
        "n_draws": samples.shape[0],
    })


def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(type(value).__name__)

In [ ]:
prediction_tables = {}
recomputed_metrics = {}
manifest_metric_deltas = []

for run_number, (label, run) in enumerate(validated.items()):
    print(f"\n[{run_number + 1}/{len(validated)}] {label}")
    run_output = OUTPUT / label
    run_output.mkdir(parents=True, exist_ok=False)
    prediction_tables[label] = {}
    recomputed_metrics[label] = {}

    transfer_group_labels = pair_groups(run["transfer"])
    populations = {
        "source_heldout": (
            run["source"],
            run["split"]["test"],
            run["split"]["source_groups"],
        ),
        "transfer_tng": (
            run["transfer"],
            run["split"]["transfer"],
            transfer_group_labels,
        ),
    }

    for population_number, (population, (frame, indices, groups)) in enumerate(populations.items()):
        selected = frame.iloc[indices]
        X = run["model"]["scaler"].transform(
            selected[run["feature_names"]].to_numpy(np.float32)
        )
        truth = selected[run["target"]].to_numpy(np.float32)
        same_host = selected["same_host"].to_numpy(bool)

        sample_seed = RANDOM_SEED + run_number * 10 + population_number
        set_sampling_seed(sample_seed)
        samples = posterior_draws(
            run["model"]["posterior"],
            X,
            DRAW_COUNT,
            EVALUATION_BATCH_SIZE,
            DEVICE,
        ).astype(np.float32, copy=False)

        metrics = metrics_by_host(samples, truth, same_host)
        recomputed_metrics[label][population] = metrics
        table = per_pair_summary(samples, frame, indices, run["target"], groups)
        prediction_tables[label][population] = table
        table.to_csv(run_output / f"{population}_posterior_summary.csv.gz", index=False)

        if SAVE_POSTERIOR_DRAWS:
            np.savez_compressed(
                run_output / f"{population}_posterior_draws.npz",
                samples=samples,
                truth=truth,
                pair_index=table["pair_index"].to_numpy(),
                same_host=same_host,
                connected_component=table["connected_component"].to_numpy(),
                seed=np.array(sample_seed),
            )

        old_metrics = run["manifest"]["evaluation"][population]
        for subset, new_values in metrics.items():
            for metric in ("rmse", "bias", "r2", "coverage68", "coverage95"):
                old_value = old_metrics[subset].get(metric)
                new_value = new_values.get(metric)
                if old_value is not None and new_value is not None:
                    manifest_metric_deltas.append({
                        "run": label,
                        "population": population,
                        "subset": subset,
                        "metric": metric,
                        "manifest_value": old_value,
                        "recomputed_value": new_value,
                        "delta": new_value - old_value,
                    })

        print(
            f"  {population:16s} n={len(indices):,} "
            f"rmse={metrics['pooled']['rmse']:.4f} "
            f"bias={metrics['pooled']['bias']:.4f}"
        )
        del samples, X
        gc.collect()
        torch.cuda.empty_cache()

    (run_output / "metrics_500_draws.json").write_text(
        json.dumps(recomputed_metrics[label], indent=2, default=json_default),
        encoding="utf-8",
    )

pd.DataFrame(manifest_metric_deltas).to_csv(
    OUTPUT / "manifest_metric_deltas.csv", index=False
)

## 5. Recomputed metric table

The table below gives the main point-prediction and calibration metrics using the common
500-draw evaluation. R2 is included for pooled and same-host reporting, but it should not be
used to rank the 277 different-host TNG pairs because that subset has a narrow target range.


In [ ]:
metric_rows = []
for label, populations in recomputed_metrics.items():
    target_name = validated[label]["config"]["target"]
    mass_feature = validated[label]["config"]["mass_feature"]
    for population, subsets in populations.items():
        for subset, values in subsets.items():
            metric_rows.append({
                "run": label,
                "target": target_name,
                "mass_feature": mass_feature,
                "population": population,
                "subset": subset,
                **values,
            })

metric_table = pd.DataFrame(metric_rows)
metric_table.to_csv(OUTPUT / "recomputed_metrics_500_draws.csv", index=False)

display_columns = [
    "run", "target", "population", "subset", "n", "rmse", "bias", "r2",
    "coverage68", "coverage95", "rank_cdf_distance",
]
metric_table.loc[:, display_columns].round(4)

## 6. Paired connected-host bootstrap

The bootstrap resamples connected host components rather than individual rows. Every selected
component is included with all of its pairs, and the same resampled components are used for
both models in a comparison. This preserves the pair dependence and makes the comparison
paired.

For M200c, the historical bound-subhalo-ratio model is the reference. For the combined
bound-subhalo-mass target, pure dynamics is the reference because a bound-mass-derived input
would leak target information. A negative RMSE difference means that the candidate has a
lower error than the reference. For bias, the absolute-bias difference is also reported because
a signed bias difference alone does not indicate an improvement.

These intervals are conditional on the fitted models and the selected catalogue populations.
They do not include uncertainty from retraining, simulation volume, or cosmic variance.


In [ ]:
COMPARISONS = {
    "m200c_bound_ratio": [
        "m200c_dynamics",
        "m200c_stellar_individual",
        "m200c_stellar_ratio",
        "m200c_stellar_sum",
    ],
    "bound_sum_dynamics": [
        "bound_sum_stellar_individual",
        "bound_sum_stellar_ratio",
        "bound_sum_stellar_sum",
    ],
}


def aligned_tables(reference: pd.DataFrame, candidate: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "pair_index", "connected_component", "same_host", "truth", "residual",
        "q025", "q16", "q84", "q975",
    ]
    merged = reference[columns].merge(
        candidate[columns], on="pair_index", suffixes=("_reference", "_candidate"),
        validate="one_to_one",
    )
    if len(merged) != len(reference) or len(merged) != len(candidate):
        raise RuntimeError("The paired tables do not contain the same evaluation rows.")
    if not np.allclose(merged["truth_reference"], merged["truth_candidate"], atol=1e-6):
        raise RuntimeError("Truth values disagree in a paired comparison.")
    if not np.array_equal(
        merged["connected_component_reference"], merged["connected_component_candidate"]
    ):
        raise RuntimeError("Connected host components disagree in a paired comparison.")
    if not np.array_equal(merged["same_host_reference"], merged["same_host_candidate"]):
        raise RuntimeError("Host membership disagrees in a paired comparison.")
    return merged


def aggregate_by_component(frame: pd.DataFrame) -> dict[str, np.ndarray]:
    group_code, _ = pd.factorize(frame["connected_component_reference"], sort=True)
    n_groups = group_code.max() + 1
    truth = frame["truth_reference"].to_numpy()
    residual_reference = frame["residual_reference"].to_numpy()
    residual_candidate = frame["residual_candidate"].to_numpy()

    values = {
        "count": np.bincount(group_code, minlength=n_groups).astype(np.float64),
        "sse_reference": np.bincount(
            group_code, weights=residual_reference**2, minlength=n_groups
        ),
        "sse_candidate": np.bincount(
            group_code, weights=residual_candidate**2, minlength=n_groups
        ),
        "residual_reference": np.bincount(
            group_code, weights=residual_reference, minlength=n_groups
        ),
        "residual_candidate": np.bincount(
            group_code, weights=residual_candidate, minlength=n_groups
        ),
    }
    for level, low, high in ((68, "q16", "q84"), (95, "q025", "q975")):
        for role in ("reference", "candidate"):
            covered = (
                (truth >= frame[f"{low}_{role}"].to_numpy())
                & (truth <= frame[f"{high}_{role}"].to_numpy())
            )
            values[f"covered{level}_{role}"] = np.bincount(
                group_code, weights=covered.astype(float), minlength=n_groups
            )
    return values


def values_from_sums(sums: dict[str, np.ndarray]) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    count = sums["count"]
    rmse_reference = np.sqrt(sums["sse_reference"] / count)
    rmse_candidate = np.sqrt(sums["sse_candidate"] / count)
    bias_reference = sums["residual_reference"] / count
    bias_candidate = sums["residual_candidate"] / count
    coverage68_reference = sums["covered68_reference"] / count
    coverage68_candidate = sums["covered68_candidate"] / count
    coverage95_reference = sums["covered95_reference"] / count
    coverage95_candidate = sums["covered95_candidate"] / count
    return {
        "rmse": (rmse_reference, rmse_candidate),
        "bias": (bias_reference, bias_candidate),
        "abs_bias": (np.abs(bias_reference), np.abs(bias_candidate)),
        "coverage68": (coverage68_reference, coverage68_candidate),
        "coverage95": (coverage95_reference, coverage95_candidate),
        "coverage_error68": (
            np.abs(coverage68_reference - 0.68),
            np.abs(coverage68_candidate - 0.68),
        ),
        "coverage_error95": (
            np.abs(coverage95_reference - 0.95),
            np.abs(coverage95_candidate - 0.95),
        ),
    }


def paired_component_bootstrap(
    frame: pd.DataFrame,
    repeats: int,
    seed: int,
    chunk_size: int,
) -> tuple[dict[str, float], dict[str, np.ndarray], int]:
    aggregate = aggregate_by_component(frame)
    n_groups = len(aggregate["count"])
    rng = np.random.default_rng(seed)

    original_sums = {key: np.array([value.sum()]) for key, value in aggregate.items()}
    original_values = values_from_sums(original_sums)
    estimates = {
        metric: float(candidate[0] - reference[0])
        for metric, (reference, candidate) in original_values.items()
    }

    draws = {metric: [] for metric in original_values}
    for start in range(0, repeats, chunk_size):
        size = min(chunk_size, repeats - start)
        chosen = rng.integers(0, n_groups, size=(size, n_groups))
        sampled_sums = {
            key: value[chosen].sum(axis=1) for key, value in aggregate.items()
        }
        sampled_values = values_from_sums(sampled_sums)
        for metric, (reference, candidate) in sampled_values.items():
            draws[metric].append(candidate - reference)

    draws = {metric: np.concatenate(parts) for metric, parts in draws.items()}
    return estimates, draws, n_groups

In [ ]:
bootstrap_rows = []
bootstrap_draws = {}
subset_masks = {
    "pooled": lambda frame: np.ones(len(frame), dtype=bool),
    "same_host": lambda frame: frame["same_host_reference"].to_numpy(bool),
    "different_host": lambda frame: ~frame["same_host_reference"].to_numpy(bool),
}

comparison_number = 0
for reference_label, candidate_labels in COMPARISONS.items():
    for candidate_label in candidate_labels:
        for population in ("source_heldout", "transfer_tng"):
            merged = aligned_tables(
                prediction_tables[reference_label][population],
                prediction_tables[candidate_label][population],
            )
            for subset, mask_function in subset_masks.items():
                selected = merged.loc[mask_function(merged)].reset_index(drop=True)
                seed = RANDOM_SEED + 1000 + comparison_number
                estimates, draws, n_groups = paired_component_bootstrap(
                    selected,
                    repeats=BOOTSTRAP_REPEATS,
                    seed=seed,
                    chunk_size=BOOTSTRAP_CHUNK_SIZE,
                )
                key_base = f"{reference_label}__{candidate_label}__{population}__{subset}"
                for metric, values in draws.items():
                    low, median, high = np.quantile(values, [0.025, 0.5, 0.975])
                    bootstrap_rows.append({
                        "reference": reference_label,
                        "candidate": candidate_label,
                        "population": population,
                        "subset": subset,
                        "metric": metric,
                        "n_pairs": len(selected),
                        "n_components": n_groups,
                        "estimate_delta": estimates[metric],
                        "bootstrap_median_delta": median,
                        "ci95_low": low,
                        "ci95_high": high,
                        "fraction_below_zero": np.mean(values < 0),
                        "seed": seed,
                    })
                    bootstrap_draws[f"{key_base}__{metric}"] = values.astype(np.float32)
                comparison_number += 1

bootstrap_summary = pd.DataFrame(bootstrap_rows)
bootstrap_summary.to_csv(OUTPUT / "paired_component_bootstrap_summary.csv", index=False)
np.savez_compressed(OUTPUT / "paired_component_bootstrap_draws.npz", **bootstrap_draws)

key_result = bootstrap_summary.loc[
    (bootstrap_summary["reference"] == "m200c_bound_ratio")
    & (bootstrap_summary["candidate"] == "m200c_dynamics")
    & (bootstrap_summary["population"] == "transfer_tng")
    & (bootstrap_summary["subset"] == "different_host")
    & (bootstrap_summary["metric"].isin(["rmse", "abs_bias", "coverage_error68"]))
]
key_result

## 7. Bootstrap comparison figure

The figure reports candidate-minus-reference RMSE on TNG. Values to the left of zero favour
the candidate. The intervals describe variation from resampling connected host components;
they should not be interpreted as including model-refit uncertainty.


In [ ]:
plot_data = bootstrap_summary.loc[
    (bootstrap_summary["population"] == "transfer_tng")
    & (bootstrap_summary["metric"] == "rmse")
].copy()
plot_data["comparison"] = (
    plot_data["candidate"].str.replace("m200c_", "", regex=False)
    .str.replace("bound_sum_", "", regex=False)
    + " vs "
    + plot_data["reference"].str.replace("m200c_", "", regex=False)
    .str.replace("bound_sum_", "", regex=False)
)

subsets = ["pooled", "same_host", "different_host"]
fig, axes = plt.subplots(1, 3, figsize=(16, 7), sharey=True)
comparisons = list(dict.fromkeys(plot_data["comparison"]))
y_lookup = {name: i for i, name in enumerate(comparisons)}

for ax, subset in zip(axes, subsets):
    current = plot_data.loc[plot_data["subset"] == subset]
    for _, row in current.iterrows():
        y = y_lookup[row["comparison"]]
        x = row["estimate_delta"]
        ax.errorbar(
            x,
            y,
            xerr=[[x - row["ci95_low"]], [row["ci95_high"] - x]],
            fmt="o",
            color="tab:blue",
            capsize=3,
        )
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
    ax.set_title(subset.replace("_", " ").title())
    ax.set_xlabel("Candidate - reference RMSE [dex]")
    ax.grid(axis="x", alpha=0.25)

axes[0].set_yticks(range(len(comparisons)))
axes[0].set_yticklabels(comparisons)
fig.suptitle("Paired connected-host bootstrap on TNG", y=0.98)
fig.tight_layout()
fig.savefig(OUTPUT / "tng_paired_rmse_differences.png", dpi=220, bbox_inches="tight")
fig.savefig(OUTPUT / "tng_paired_rmse_differences.pdf", bbox_inches="tight")
plt.show()

## 8. Output manifest

The output manifest records the input runs, package versions, evaluation seeds, and all files
created by the notebook. The results remain conditional on the fitted models. The next
analysis should use these per-pair exports together with held-out domain-classifier scores to
test whether the transfer bias remains inside common feature support.


In [ ]:
output_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project": str(PROJECT),
    "output": str(OUTPUT),
    "purpose": "Evaluation-only export and paired connected-host bootstrap; no NPE training",
    "device": torch.cuda.get_device_name(0),
    "versions": actual_versions,
    "settings": {
        "posterior_draws": DRAW_COUNT,
        "evaluation_batch_size": EVALUATION_BATCH_SIZE,
        "bootstrap_repeats": BOOTSTRAP_REPEATS,
        "random_seed": RANDOM_SEED,
        "save_posterior_draws": SAVE_POSTERIOR_DRAWS,
    },
    "runs": {label: str(path) for label, path in RUNS.items()},
    "outputs": sorted(
        str(path.relative_to(OUTPUT)) for path in OUTPUT.rglob("*") if path.is_file()
    ),
    "reporting_notes": [
        "R2 is not used to rank the 277 different-host TNG pairs because their target range is narrow.",
        "Bootstrap intervals are conditional on the fitted models and do not include retraining or cosmic-variance uncertainty.",
        "Posterior metrics were recomputed with 500 draws and can differ slightly from the 200-draw run manifests.",
    ],
}
(OUTPUT / "evaluation_manifest.json").write_text(
    json.dumps(output_manifest, indent=2), encoding="utf-8"
)

print(f"Completed evaluation export: {OUTPUT}")
print("No model was retrained.")